# Week 02: Mini Semantic Search

## Introduction

Semantic search is a technique that finds information based on meaning rather than only matching exact words.

In this project, we build a mini semantic search system for frequently asked questions (FAQs). The system converts questions and answers into numerical representations called embeddings and stores them in ChromaDB.

When a user enters a query, the system searches for FAQs with similar meanings and returns the three most relevant results.

## Objectives

* Load FAQ data from a CSV file using Pandas.
* Clean the dataset.
* Combine questions and answers into text chunks.
* Generate embeddings using Sentence Transformers.
* Store embeddings in ChromaDB.
* Search for the three most relevant FAQs.
* Calculate basic statistics using NumPy.

## Tools and Libraries

* Python
* Pandas
* NumPy
* Sentence Transformers
* ChromaDB
* Google Colab

## Dataset

The dataset contains frequently asked questions and their answers. It should contain 30 FAQs before cleaning.


In [21]:
%pip install -q pandas numpy sentence-transformers chromadb

## Step 1: Import Libraries

We import the libraries needed for this project.

* **Pandas:** Loads and processes the FAQ dataset.
* **NumPy:** Calculates numerical statistics.
* **Sentence Transformers:** Converts text into embeddings.
* **ChromaDB:** Stores embeddings and retrieves relevant FAQs.


In [22]:
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Load the Dataset

We load the FAQ dataset from a CSV file using Pandas.

The `read_csv()` function reads the CSV file into a DataFrame. A DataFrame is a table containing rows and columns.

We display the first five rows to inspect the dataset and count the FAQs.


In [23]:
df = pd.read_csv("/content/faqs.csv")

print("Number of FAQs:", len(df))
display(df.head())

Number of FAQs: 30


,id,question,answer
0,1,What is Python?,Python is a programming language used for data...
1,2,What is machine learning?,Machine learning is a type of AI where compute...
2,3,What is a DataFrame?,A DataFrame is a table-like structure in panda...
3,4,What is Git?,Git is a version control system that tracks ch...
4,5,What is an API?,An API is a way for two programs to communicat...


## Step 3: Clean the Dataset

Data cleaning prepares the dataset for processing.

We remove rows with missing questions or answers, remove extra spaces, remove empty questions and answers, and remove duplicate questions.

This helps prevent incomplete or repeated information from affecting the search results.


In [24]:
print("FAQs before cleaning:", len(df))

df = df.dropna(subset=["question", "answer"]).copy()

df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates(subset=["question"])
df = df.reset_index(drop=True)

print("FAQs after cleaning:", len(df))
display(df.head())

FAQs before cleaning: 30
FAQs after cleaning: 25


,id,question,answer
0,1,What is Python?,Python is a programming language used for data...
1,2,What is machine learning?,Machine learning is a type of AI where compute...
2,3,What is a DataFrame?,A DataFrame is a table-like structure in panda...
3,4,What is Git?,Git is a version control system that tracks ch...
4,5,What is an API?,An API is a way for two programs to communicat...


## Step 4: Create Text Chunks

A text chunk is a piece of text that the search system processes.

For this project, each chunk contains one question and its answer. Combining them gives the search system more information about the meaning of each FAQ.



In [25]:
df["chunk"] = (
    "Question: " + df["question"] +
    " Answer: " + df["answer"]
)

chunks = df["chunk"].tolist()

print("Number of text chunks:", len(chunks))
print("\nExample text chunk:")
print(chunks[0])

Number of text chunks: 25

Example text chunk:
Question: What is Python? Answer: Python is a programming language used for data science and web development.


## Step 5: Generate Text Embeddings

Embeddings are numerical representations of text.

The `all-MiniLM-L6-v2` model converts each text chunk into a vector. Texts with similar meanings can have similar vector representations.

These embeddings allow us to search for relevant FAQs even when the user's words differ from the original question.



In [26]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of embeddings: 25
Embedding shape: (25, 384)


## Step 6: Store Embeddings in ChromaDB

ChromaDB is a vector database that stores embeddings and helps retrieve relevant information.

We create a collection named `faqs`. We store each text chunk, its embedding, and its original question and answer.

The original question and answer are saved as metadata so we can display them when a search returns a match.


In [34]:
client = chromadb.Client()

# Reuse the collection if it already exists.
collection = client.get_or_create_collection(name="faqs")

# Clear previous records so we can store the current dataset.
existing_ids = collection.get()["ids"]

if existing_ids:
    collection.delete(ids=existing_ids)

# Store the current FAQ chunks and embeddings.
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=[
        {"question": q, "answer": a}
        for q, a in zip(df["question"], df["answer"])
    ]
)

print("Documents stored in ChromaDB:", collection.count())

Documents stored in ChromaDB: 25


## Step 7: Create the Search Function

The search function receives a user's query and converts it into an embedding using the same model.

ChromaDB compares the query embedding with the stored embeddings and retrieves the three most relevant FAQs.

The function displays the matching questions and answers.


In [32]:
def search_faqs(query, n=3):
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(n, collection.count())
    )

    print(f"\nQuery: {query}")
    print("-" * 60)

    for i, item in enumerate(results["metadatas"][0], start=1):
        print(f"{i}. {item['question']}")
        print(f"   Answer: {item['answer']}")
        print()

    return results

## Step 8: Test Semantic Search

We test the search system with different queries.

The queries use different wording to check whether the system can retrieve relevant information based on meaning rather than requiring an exact keyword match.


In [30]:
search_faqs("How do I track changes in my code?")

search_faqs("What tool helps applications run everywhere?")

search_faqs("How can I find patterns in data?")

search_faqs("How can I search for information by meaning?")

search_faqs("Which Python library helps me work with arrays?")


Query: How do I track changes in my code?
------------------------------------------------------------
1. What is version control?
   Answer: Version control tracks changes to files over time and helps developers manage different versions of a project.

2. What is Git?
   Answer: Git is a version control system that tracks changes in code over time.

3. What is a function?
   Answer: A function is a reusable block of code that performs a specific task.


Query: What tool helps applications run everywhere?
------------------------------------------------------------
1. What is Docker?
   Answer: Docker is a tool that packages applications into containers.

2. What is an API used for?
   Answer: An API allows different software applications to communicate and exchange information.

3. What is version control?
   Answer: Version control tracks changes to files over time and helps developers manage different versions of a project.


Query: How can I find patterns in data?
----------------

{'ids': [['17', '0', '18']],
 'embeddings': None,
 'documents': [['Question: What is NumPy? Answer: NumPy is a Python library used for numerical calculations and working with arrays.',
   'Question: What is Python? Answer: Python is a programming language used for data science and web development.',
   'Question: What is Pandas? Answer: Pandas is a Python library used to organize, clean, analyze, and manipulate structured data.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'question': 'What is NumPy?',
    'answer': 'NumPy is a Python library used for numerical calculations and working with arrays.'},
   {'answer': 'Python is a programming language used for data science and web development.',
    'question': 'What is Python?'},
   {'question': 'What is Pandas?',
    'answer': 'Pandas is a Python library used to organize, clean, analyze, and manipulate structured data.'}]],
 'distances': [[0.8673586845397949, 1.198601484298706, 

## Step 9: Calculate Basic Statistics

We use NumPy to calculate statistics about the lengths of the FAQ questions.

We calculate:

* The average question length.
* The median question length.
* The maximum question length.
* The 75th percentile.

These statistics help us understand the size and distribution of the questions in our dataset.


In [31]:
lengths = df["question"].str.len().to_numpy()

print("Number of FAQs:", len(df))
print(f"Average question length: {np.mean(lengths):.1f}")
print(f"Median question length: {np.median(lengths):.1f}")
print(f"Maximum question length: {np.max(lengths)}")
print(f"75th percentile: {np.percentile(lengths, 75):.1f}")

Number of FAQs: 25
Average question length: 20.3
Median question length: 21.0
Maximum question length: 36
75th percentile: 24.0


## Conclusion

In this project, I developed a mini semantic search system for FAQs.

I loaded and cleaned the dataset, combined questions and answers into text chunks, generated embeddings using Sentence Transformers, and stored them in ChromaDB.

I then tested the system with different queries to retrieve relevant FAQs based on meaning. Finally, I used NumPy to calculate statistics about question lengths.

The dataset contains 30 FAQs before cleaning, provided that the updated CSV file was uploaded successfully. The final number of FAQs may be lower after cleaning.

This project demonstrates how embeddings and vector databases can support semantic search.
